# 📄 Chat with your PDF (RAG from scratch)

Upload any PDF and ask questions about it. The AI answers **only from your PDF** and tells you the **page number**.

**What we build, in 5 steps** (same order as the HTML pages in `RAG_Agents_Visualized/`):

| Step | What happens | HTML page |
|---|---|---|
| 1 | Read the PDF into text | `02_pdf_to_chunks.html` |
| 2 | Cut the text into small chunks | `02_pdf_to_chunks.html` |
| 3 | Turn every chunk into numbers (embeddings) | `03_embeddings.html` |
| 4 | Find the chunks closest to the question | `04_search_and_answer.html` |
| 5 | Give those chunks + the question to the LLM | `04_search_and_answer.html` |

**Tools (all free):** `pypdf` reads the PDF · `sentence-transformers` (open-source, runs on your laptop) makes embeddings · `numpy` searches · **Groq** runs the open-source **Llama 3.3** model in the cloud.

> 🔑 You need a free Groq key. See `RAG_Agents_Visualized/00_setup.html`.

## 0. Install the libraries (run once)

Same exact versions as `requirements.txt`. On Colab, if it asks you to **restart the session**, do it and continue from the next cell.

In [ ]:
%pip install -q groq==1.7.0 pypdf==6.19.0 sentence-transformers==6.1.0

## 0. Load your API key(s)

The cell looks for each key in this order:
1. **Colab:** the 🔑 **Secrets** panel on the left (add the key there once, with notebook access switched on).
2. **Laptop:** an environment variable (see `RAG_Agents_Visualized/00_setup.html`).
3. Otherwise it **asks you** to paste it.

**Never type a key directly into the code.** It would end up on GitHub, and anyone could use it.

In [ ]:
import os
from getpass import getpass

def load_key(name):
    if os.environ.get(name):                      # laptop: environment variable
        return
    try:
        from google.colab import userdata         # Colab: 🔑 Secrets panel
        os.environ[name] = userdata.get(name)
    except Exception:
        os.environ[name] = getpass(f"Paste your {name}: ")   # otherwise: ask

load_key("GROQ_API_KEY")
print("Keys loaded ✅")

## 1. Choose a PDF

- **On Colab:** a file picker appears. Choose any PDF (try `acme_handbook.pdf` from the `data` folder: a fictional company handbook).
- **On a laptop:** it uses `data/acme_handbook.pdf`. Change the path to use your own PDF.

*Use a PDF with real text. A scanned photo of a page has no text to read.*

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:                # on Colab: a file picker appears below
    from google.colab import files
    uploaded = files.upload()
    pdf_path = Path(list(uploaded)[0])
else:                                            # on a laptop: put your PDF's path here
    pdf_path = Path("data/acme_handbook.pdf")

print("Using:", pdf_path)

## 1. Read the PDF into text

`pypdf` gives us the text of each page. We keep pages separate so we can say *which page* an answer came from.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pages = [page.extract_text() or "" for page in reader.pages]

print(f"{len(pages)} pages, {sum(len(p) for p in pages):,} characters")
print("\n--- start of page 1 ---\n")
print(pages[0][:500])

## 2. Cut the text into chunks

An LLM can't read 300 pages at once, and a small piece is easier to match to a question.
So we cut each page into **chunks of 500 characters**.

Neighbouring chunks **share 100 characters (the overlap)**, so a sentence cut at the border still appears whole in one of them.

In [ ]:
CHUNK_SIZE = 500   # characters per chunk
OVERLAP = 100      # characters shared by neighbouring chunks

chunks = []        # each chunk remembers its text and its page number
for page_no, text in enumerate(pages, start=1):
    text = " ".join(text.split())               # tidy up spaces and line breaks
    start = 0
    while True:
        chunks.append({"text": text[start:start + CHUNK_SIZE], "page": page_no})
        if start + CHUNK_SIZE >= len(text):      # reached the end of the page
            break
        start += CHUNK_SIZE - OVERLAP

print(len(chunks), "chunks\n")
for c in chunks[:4]:
    print(f"[page {c['page']}] {c['text'][:90]}…")

## 3. Turn every chunk into numbers (embeddings)

`all-MiniLM-L6-v2` is a small open-source model (about 90 MB, downloaded once). It turns any text into **384 numbers**.
Texts with **similar meaning** get **similar numbers**, even when they use different words.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

print("shape:", chunk_vectors.shape, "  → (number of chunks, 384 numbers each)")
print("first 8 numbers of chunk 0:", chunk_vectors[0][:8].round(3))

### 👀 See it work: similar meaning → high score

The score is between 0 and 1: **close to 1 = same meaning**, **close to 0 = unrelated**.

In [ ]:
question, leave, laptop = embedder.encode(
    ["How many holidays do I get?",
     "Every employee gets 18 days of annual leave per year.",
     "Laptops are replaced every 3 years."],
    normalize_embeddings=True)

print("question vs leave sentence :", round(float(question @ leave), 2))
print("question vs laptop sentence:", round(float(question @ laptop), 2))

## 4. Search: find the chunks closest to the question

1. Turn the question into 384 numbers.
2. Compare it with **every** chunk (one line of numpy).
3. Keep the **top 3**.

For a few thousand chunks, plain numpy is fast enough. Big apps use a vector database (FAISS, Chroma) for the same job.

In [ ]:
import numpy as np

def search(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q                   # similarity with every chunk
    best = np.argsort(scores)[::-1][:k]          # positions of the k highest scores
    return [(float(scores[i]), chunks[i]) for i in best]

for score, c in search("How many sick days do I get?"):
    print(f"{score:.2f} | page {c['page']} | {c['text'][:90]}…")

## 5. Ask the LLM, using only those chunks

We build a **prompt** = rules + the 3 chunks + the question, and send it to Llama 3.3 on Groq.

The rule *"answer ONLY from the context, otherwise say I don't know"* is what stops the model from making things up.

In [ ]:
from groq import Groq

client = Groq()                          # reads GROQ_API_KEY automatically
MODEL = "llama-3.3-70b-versatile"        # open-source Llama 3.3 on Groq (model list: console.groq.com/docs/models)

def ask(question, k=3, show_prompt=False):
    hits = search(question, k)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for _, c in hits)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the document."
Mention the page number you used, like (page 2).

Context:
{context}

Question: {question}"""

    if show_prompt:
        print(prompt, "\n" + "-" * 60)

    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,                   # 0 = no creativity, stick to the facts
    )
    return reply.choices[0].message.content

print(ask("How many leave days do I get after 6 years?", show_prompt=True))

### 🧪 Try more questions (the last one is not in the PDF on purpose)

In [ ]:
for q in ["Can I work from home on Fridays?",
          "What is the notice period if I resign?",
          "What is the dress code?"]:
    print("❓", q)
    print("🤖", ask(q), "\n")

## 🎉 Your PDF chatbot

Run the cell, type a question in the box and press Enter. Type `bye` to stop.
Pick a different PDF in step 1, re-run the cells, and it works on that too.

In [ ]:
while True:
    question = input("❓ Ask your PDF (or type bye): ")
    if question.strip().lower() in ("bye", "exit", ""):
        break
    print("🤖", ask(question), "\n")

---
## 🧠 Recap: the whole RAG in one sentence

**Cut the PDF into chunks → turn them into numbers → find the chunks closest to the question → let the LLM answer from only those chunks.**

## 🛠️ Make it your own (your project)

Copy this notebook and change **one thing at a time**:

| Project idea | What you change |
|---|---|
| 📚 **Syllabus / notes helper**: ask questions about your course notes | Upload your notes PDF |
| 📝 **Resume Q&A**: a recruiter can ask "Does this candidate know Docker?" | Upload your resume; change the rules in the prompt |
| 🔬 **Research paper explainer**: "Explain this paper like I'm 15" | Add *"explain simply"* to the prompt |
| ⚖️ **Rules & policies bot**: college rules, rent agreement, insurance policy | Upload the document; ask real questions |
| 📂 **Many PDFs at once** | Loop over several files; store the file name in each chunk |

**Experiments to try (and write down what happens):**
1. Change `CHUNK_SIZE` to 200 and to 1000. Do the answers get better or worse?
2. Change `k` from 3 to 1. When does the answer break?
3. Remove the "ONLY from the context" rule and ask about the dress code. What happens?

**To put it on your resume:**
- Push it to GitHub **without your key** (keep it in the environment variable, never in the code).
- Write a short README: what it does, a screenshot, how to run it.
- Resume line: *Built a RAG system to chat with PDFs (Python, sentence-transformers, Groq API, Llama 3.3).*